In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Загружаем данные
docs = pd.read_csv('/home/ruslan/RAG_legal_documents/data/documents.csv')
train = pd.read_csv('/home/ruslan/RAG_legal_documents/data/train.csv')
test = pd.read_csv('/home/ruslan/RAG_legal_documents/data/test.csv')

print(f"Документов: {len(docs)}")
print(f"Train: {len(train)}")
print(f"Test: {len(test)}")
print(f"Уникальных правильных документов в train: {train['gold_doc_id'].nunique()}")

Документов: 468
Train: 700
Test: 350
Уникальных правильных документов в train: 113


In [3]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
nltk.download('punkt')
nltk.download('stopwords')

# Стоп-слова
RUSSIAN_STOPWORDS = set(stopwords.words('russian'))

# Дополнительные стоп-слова из бейзлайна
EXTRA_STOP = set('и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему когда даже ну ли если уже или ни быть был него до вас уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам без чего раз тоже себе под будет тогда кто этот того потому этого какой ним здесь этом один мой тем чтобы нее были куда зачем всех при два об другой хоть после над больше тот через эти нас про всего них какая много три эту перед лучше том такой им более всю между'.split())
STOPWORDS = RUSSIAN_STOPWORDS.union(EXTRA_STOP)

def tokenize(text):
    """Токенизация текста для BM25"""
    if not isinstance(text, str):
        text = str(text)
    # Только буквы и цифры, нижний регистр
    tokens = re.findall(r'[а-яёa-z0-9]+', text.lower())
    # Фильтр по длине и стоп-словам
    return [t for t in tokens if len(t) > 2 and t not in STOPWORDS]

# Проверка
sample = "Признаки помогут доказать, что заявление не подписывалось"
print(tokenize(sample))

['признаки', 'помогут', 'доказать', 'заявление', 'подписывалось']


[nltk_data] Downloading package punkt to /home/ruslan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/ruslan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
from rank_bm25 import BM25Okapi

# Токенизируем все документы
print("Токенизация документов для BM25...")
tokenized_docs = [tokenize(text) for text in tqdm(docs['text'])]

# Создаём BM25 индекс
bm25 = BM25Okapi(tokenized_docs)
doc_ids = docs['doc_id'].tolist()

print(f"BM25 индекс создан. Документов: {len(doc_ids)}")

Токенизация документов для BM25...


  0%|          | 0/468 [00:00<?, ?it/s]

100%|██████████| 468/468 [00:00<00:00, 2414.44it/s]


BM25 индекс создан. Документов: 468


In [5]:
from sentence_transformers import SentenceTransformer

# Загружаем модель
# Используем small модель для скорости
model_name = 'intfloat/multilingual-e5-small'
print(f"Загрузка {model_name}...")
model = SentenceTransformer(model_name)

print(f"Модель загружена. Размер эмбеддинга: {model.get_sentence_embedding_dimension()}")

# Тест
test_text = "Признаки помогут доказать, что заявление не подписывалось"
test_emb = model.encode(test_text)
print(f"Тестовый эмбеддинг размер: {len(test_emb)}")

Загрузка intfloat/multilingual-e5-small...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 799.08it/s]


Модель загружена. Размер эмбеддинга: 384
Тестовый эмбеддинг размер: 384


In [7]:
import faiss
import numpy as np
from tqdm import tqdm

# Кодируем все документы
print("Кодирование документов...")
doc_embeddings = []
batch_size = 32

for i in tqdm(range(0, len(docs), batch_size)):
    batch = docs['text'].iloc[i:i+batch_size].tolist()
    batch_embs = model.encode(batch, show_progress_bar=False)
    doc_embeddings.append(batch_embs)

doc_embeddings = np.vstack(doc_embeddings).astype('float32')
print(f"Матрица эмбеддингов: {doc_embeddings.shape}")

# Создаём FAISS индекс
dim = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner Product = косинус для нормированных векторов
faiss.normalize_L2(doc_embeddings)  # Нормируем для косинуса
index.add(doc_embeddings)

print(f"FAISS индекс создан. Документов: {index.ntotal}")

Кодирование документов...


  0%|          | 0/15 [00:00<?, ?it/s]

100%|██████████| 15/15 [02:11<00:00,  8.75s/it]

Матрица эмбеддингов: (468, 384)
FAISS индекс создан. Документов: 468


In [8]:
def dense_search(query, top_k=20):
    """Поиск по эмбеддингам"""
    # Кодируем запрос
    query_emb = model.encode(query)
    query_emb = query_emb.reshape(1, -1).astype('float32')
    faiss.normalize_L2(query_emb)
    
    # Ищем
    scores, indices = index.search(query_emb, top_k)
    
    # Возвращаем doc_id и скоры
    results = [(doc_ids[idx], float(scores[0][i])) for i, idx in enumerate(indices[0])]
    return results

# Тест
test_results = dense_search("заявление о переходе в НПФ не подписывалось", top_k=5)
for doc_id, score in test_results:
    print(f"{doc_id}: {score:.4f}")

d_f8787907b3: 0.8772
d_4d2febd25e: 0.8763
d_de01daec14: 0.8762
d_ca79b1b741: 0.8757
d_b2f62b4d48: 0.8749


In [9]:
def bm25_search(query, top_k=20):
    """Поиск по BM25"""
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    
    # Сортируем
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = [(doc_ids[i], float(scores[i])) for i in top_indices]
    return results

# Тест
test_results = bm25_search("заявление о переходе в НПФ не подписывалось", top_k=5)
for doc_id, score in test_results:
    print(f"{doc_id}: {score:.4f}")

d_8357734cfe: 11.3716
d_ede31c25ba: 10.2715
d_3b12d1fbd6: 9.2150
d_a3ea97ebfc: 7.3615
d_fac20a2a71: 7.2797


In [10]:
def hybrid_search(query, top_k=5):
    """Гибридный поиск: BM25 + Dense"""
    # Получаем по 20 кандидатов от каждого метода
    bm25_results = bm25_search(query, top_k=20)
    dense_results = dense_search(query, top_k=20)
    
    # Reciprocal Rank Fusion
    scores = {}
    for rank, (doc_id, _) in enumerate(bm25_results):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (rank + 1)
    for rank, (doc_id, _) in enumerate(dense_results):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (rank + 1)
    
    # Сортируем по сумме
    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in sorted_docs[:top_k]]

# Тест
test_query = "заявление о переходе в НПФ не подписывалось"
hybrid_results = hybrid_search(test_query, top_k=5)
print(f"Гибридный поиск: {hybrid_results}")

Гибридный поиск: ['d_8357734cfe', 'd_f8787907b3', 'd_ede31c25ba', 'd_4d2febd25e', 'd_3b12d1fbd6']


In [11]:
from tqdm import tqdm

def recall_at_5(gold_list, preds_list):
    """Расчёт Recall@5"""
    hits = 0
    for gold, pred in zip(gold_list, preds_list):
        if gold in pred[:5]:
            hits += 1
    return hits / len(gold_list)

# Тестируем на первых 100 вопросах train (для скорости)
sample_size = 100
train_sample = train.head(sample_size)

print("Поиск для sample train...")
preds = []
for question in tqdm(train_sample['question']):
    preds.append(hybrid_search(question, top_k=5))

r5 = recall_at_5(train_sample['gold_doc_id'].tolist(), preds)
print(f"Recall@5 на {sample_size} вопросах: {r5:.4f}")

Поиск для sample train...


100%|██████████| 100/100 [00:04<00:00, 21.30it/s]

Recall@5 на 100 вопросах: 0.3700


In [12]:
def chunk_text(text, chunk_size=500, overlap=100):
    """Разбивка текста на чанки с перекрытием"""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

# Создаём чанки для всех документов
print("Создание чанков...")
all_chunks = []
chunk_to_doc = []  # связь чанка с doc_id

for idx, row in tqdm(docs.iterrows(), total=len(docs)):
    doc_id = row['doc_id']
    text = row['text']
    chunks = chunk_text(text, chunk_size=500, overlap=100)
    all_chunks.extend(chunks)
    chunk_to_doc.extend([doc_id] * len(chunks))

print(f"Всего чанков: {len(all_chunks)}")
print(f"Среднее чанков на документ: {len(all_chunks) / len(docs):.1f}")

Создание чанков...


100%|██████████| 468/468 [00:00<00:00, 5737.75it/s]

Всего чанков: 1274
Среднее чанков на документ: 2.7


In [13]:
# Кодируем чанки
print("Кодирование чанков...")
chunk_embeddings = []
batch_size = 32

for i in tqdm(range(0, len(all_chunks), batch_size)):
    batch = all_chunks[i:i+batch_size]
    batch_embs = model.encode(batch, show_progress_bar=False)
    chunk_embeddings.append(batch_embs)

chunk_embeddings = np.vstack(chunk_embeddings).astype('float32')
print(f"Матрица чанков: {chunk_embeddings.shape}")

# FAISS индекс для чанков
dim = chunk_embeddings.shape[1]
chunk_index = faiss.IndexFlatIP(dim)
faiss.normalize_L2(chunk_embeddings)
chunk_index.add(chunk_embeddings)

Кодирование чанков...


100%|██████████| 40/40 [05:56<00:00,  8.92s/it]

Матрица чанков: (1274, 384)


In [14]:
def dense_search_chunks(query, top_k=20):
    """Поиск по чанкам, агрегация по документам"""
    # Кодируем запрос
    query_emb = model.encode(query)
    query_emb = query_emb.reshape(1, -1).astype('float32')
    faiss.normalize_L2(query_emb)
    
    # Ищем чанки
    scores, indices = chunk_index.search(query_emb, top_k * 3)  # больше чанков
    
    # Агрегируем по документам (берём лучший чанк от документа)
    doc_scores = {}
    for i, idx in enumerate(indices[0]):
        doc_id = chunk_to_doc[idx]
        score = float(scores[0][i])
        if doc_id not in doc_scores or score > doc_scores[doc_id]:
            doc_scores[doc_id] = score
    
    # Сортируем по скору
    sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in sorted_docs[:top_k]]

# Тест
test_results = dense_search_chunks("заявление о переходе в НПФ не подписывалось", top_k=5)
print(f"Поиск по чанкам: {test_results}")

Поиск по чанкам: ['d_8357734cfe', 'd_168537aacb', 'd_2e1da709f9', 'd_435d80eff0', 'd_d68f4ab7ea']


In [15]:
def hybrid_search_chunks(query, top_k=5):
    """Гибридный поиск: BM25 по документу + Dense по чанкам"""
    # BM25 по полным документам
    bm25_results = bm25_search(query, top_k=20)
    
    # Dense по чанкам
    dense_results = dense_search_chunks(query, top_k=20)
    dense_docs = [(doc_id, 0) for doc_id in dense_results]  # dummy scores для RRF
    
    # Reciprocal Rank Fusion
    scores = {}
    for rank, (doc_id, _) in enumerate(bm25_results):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (rank + 1)
    for rank, doc_id in enumerate(dense_results):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (rank + 1)
    
    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in sorted_docs[:top_k]]

# Тестируем
sample_size = 100
train_sample = train.head(sample_size)

print("Тестирование гибридного поиска с чанками...")
preds = []
for question in tqdm(train_sample['question']):
    preds.append(hybrid_search_chunks(question, top_k=5))

r5 = recall_at_5(train_sample['gold_doc_id'].tolist(), preds)
print(f"Recall@5 с чанками: {r5:.4f}")

Тестирование гибридного поиска с чанками...


100%|██████████| 100/100 [00:04<00:00, 21.24it/s]

Recall@5 с чанками: 0.4200


In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Загружаем кросс-энкодер для русского
model_name = 'cross-encoder/ms-marco-MiniLM-L-6-v2'  # английский, но работает с транслитом
# Можно заменить на русскоязычный, если найду

print(f"Загрузка реранкера {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
reranker = AutoModelForSequenceClassification.from_pretrained(model_name)
reranker.eval()

def rerank(query, candidates, top_k=5):
    """Реранкинг кандидатов с помощью кросс-энкодера"""
    pairs = [[query, docs[docs['doc_id'] == doc_id]['text'].iloc[0][:512]] for doc_id in candidates]
    
    # Инференс
    with torch.no_grad():
        inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512)
        scores = reranker(**inputs).logits.squeeze().tolist()
    
    # Если один кандидат, scores может быть числом
    if isinstance(scores, float):
        scores = [scores]
    
    # Сортируем по скору
    sorted_pairs = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in sorted_pairs[:top_k]]

Загрузка реранкера cross-encoder/ms-marco-MiniLM-L-6-v2...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1725.59it/s]


In [ ]:
def full_pipeline(query, top_k=5):
    """Полный пайплайн: гибридный поиск → реранкер"""
    # Шаг 1: получаем 20 кандидатов
    candidates = hybrid_search_chunks(query, top_k=20)
    
    # Шаг 2: реранкинг
    reranked = rerank(query, candidates, top_k=top_k)
    
    return reranked

# Тестируем
sample_size = 100
train_sample = train.head(sample_size)

print("Тестирование полного пайплайна...")
preds = []
for question in tqdm(train_sample['question']):
    preds.append(full_pipeline(question, top_k=5))

r5 = recall_at_5(train_sample['gold_doc_id'].tolist(), preds)
print(f"Recall@5 с реранкером: {r5:.4f}")

Тестирование полного пайплайна...


  9%|▉         | 9/100 [00:25<04:12,  2.77s/it]